In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, BooleanType, TimestampType, DoubleType
)
from pyspark.sql.functions import col, from_json
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog_name", "")
catalog_name = dbutils.widgets.get("catalog_name")
dbutils.widgets.text("secret_scope", "")
SECRET_SCOPE = dbutils.widgets.get("secret_scope")
dbutils.widgets.text("evh_namespace", "")
evh_namespace = dbutils.widgets.get("evh_namespace")
dbutils.widgets.text("evh_name", "")
evh_name = dbutils.widgets.get("evh_name")

In [0]:
personal_conn_string = dbutils.secrets.get(scope=SECRET_SCOPE, key="connstr")

In [0]:
my_data = {
    "evh_namespace": evh_namespace,
    "evh_name": evh_name,
    "evh_conn_string": personal_conn_string
}

target_table = f"{catalog_name}.crime_dataset_bronze.crime_evh_bronze"
checkpoint_loc = f"/Volumes/{catalog_name}/crime_dataset_bronze/bronze/checkpoints/"

In [0]:
crime_schema = StructType([
    StructField("incident_id", StringType(), True),         
    StructField("crime_type", StringType(), True),         
    StructField("district", StringType(), True),            
    StructField("city", StringType(), True),                
    StructField("state", StringType(), True),               
    StructField("address", StringType(), True),            
    StructField("latitude", StringType(), True),   #         
    StructField("longitude", StringType(), True),   #        
    StructField("incident_datetime", TimestampType(), True),   
    StructField("officer_id", StringType(), True),          
    StructField("officer_first_name", StringType(), True), 
    StructField("officer_last_name", StringType(), True),  
    StructField("badge_number", StringType(), True),        
    StructField("suspect_id", StringType(), True),          
    StructField("suspect_first_name", StringType(), True),  
    StructField("suspect_last_name", StringType(), True),   
    StructField("suspect_age", StringType(), True), #       
    StructField("suspect_gender", StringType(), True),      
    StructField("suspect_race", StringType(), True),        
    StructField("victim_id", StringType(), True),           
    StructField("victim_first_name", StringType(), True),   
    StructField("victim_last_name", StringType(), True),    
    StructField("victim_age", StringType(), True),   #      
    StructField("victim_gender", StringType(), True),       
    StructField("victim_phone", StringType(), True),        
    StructField("weapon_used", StringType(), True),        
    StructField("severity", StringType(), True),            
    StructField("case_status", StringType(), True),         
    StructField("resolution", StringType(), True),          
    StructField("num_arrests", StringType(), True),      #  
    StructField("property_loss_usd", StringType(), True),   #
    StructField("reported_online", StringType(), True),     
    StructField("notes", StringType(), True)                
])

In [0]:
kafka_options = {
    "kafka.bootstrap.servers": f"{my_data["evh_namespace"]}.servicebus.windows.net:9093",
    "subscribe": my_data["evh_name"],
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": (
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" '
    f'password="{my_data["evh_conn_string"]}";'
    ),    
    "startingOffsets": "earliest"
}

In [0]:
df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)


parsed_df = (df
    .withColumn("value", col("value").cast("string"))
    .withColumn("value", from_json(col("value"), crime_schema))    
)

In [0]:
parsed_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_loc) \
    .trigger(availableNow=True)\
    .toTable(target_table)